In [ ]:
%matplotlib widget
import os
import json
import joblib
from pathlib import Path
import torch, einops
import matplotlib.pyplot as plt
import numpy as np

from transformer import (
    configure_logging,
    download_and_concat,
    textfile_to_tokens_as_binary,
    run_training,
    train_bpe,
    Tokenizer,
    TransformerLM,
    AdamW,
    lr_cosine_schedule,
)


configure_logging()  # or configure_logging(logging.DEBUG) for verbose

In [ ]:
# tokenize this text
special_tokens = ["<|endoftext|>", "<|begin|>", "<|end|>"]
path = "data/TinyStoriesV2-GPT4-valid.txt"
vocabulary_size = 5000

tokenizer_dir = Path("tokenizer")
tokenizer_dir.mkdir(exist_ok=True)
tokenizer_path = tokenizer_dir / "tokenizer.joblib"

if tokenizer_path.exists():
    tokenizer = joblib.load(tokenizer_path)
else:
    vocab, merges = train_bpe(path, vocabulary_size, special_tokens)
    tokenizer = Tokenizer(vocab, merges, special_tokens)
    joblib.dump(tokenizer, tokenizer_path)

In [ ]:
# download some internet text
urls = [
    "https://gutenberg.org/cache/epub/1184/pg1184.txt",
    "https://gutenberg.org/cache/epub/1513/pg1513.txt",
]
# download_and_concat(urls, "data/combined.txt", separator="\n<|endoftext|>\n")


# convert the file into a raw-binary which we can read as mmmap.
# textfile_to_tokens_as_binary(
#     "data/TinyStoriesV2-GPT4-train.txt", "data/train.bin", tokenizer, "wb"
# )
# textfile_to_tokens_as_binary(
#     "data/TinyStoriesV2-GPT4-valid.txt", "data/valid.bin", tokenizer, "wb"
# )
# os.path.getsize("data/train.bin")
# data = np.memmap("data/train.bin", dtype=np.uint16, mode="r")
# get_batch(data, 20, 5) returns 20 batches of length 5 sequences inputs,labels

In [ ]:
# run configuration -- model/optimizer classes + their exact constructor kwargs,
# plus everything else needed to train
config = {
    "description": "baseline4layerV2",
    "seed": 0,
    "model_class": TransformerLM,
    "model_params": {
        "vocab_size": vocabulary_size,
        "context_length": 128,
        "num_layers": 2,
        "d_model": 128,
        "d_ff": 1344,
        "num_heads": 1,
        "rope_theta": 10000,
        "device": "mps",
        "dtype": None,  # TODO: track this down - does this control all machine precision downstream?
    },
    "optimizer_class": torch.optim.AdamW,
    "optimizer_params": {
        "lr": 0.001,
        "betas": (0.9, 0.999),
        "weight_decay": 0.1,
        "eps": 1e-8,
    },
    "lr_schedule_fn": lr_cosine_schedule,
    "lr_schedule_params": {
        "max_learning_rate": 0.001,
        "min_learning_rate": 0.0001,
        "warmup_iters": 30,
        "cosine_cycle_iters": 300,
    },
    "training": {
        "train_path": "data/train.bin",
        "valid_path": "data/valid.bin",
        "batch_size": 64,
        "total_iterations": 5000,
        "val_every": 10,
        "save_every": 50,
    },
}

In [ ]:
# starts a brand-new run. to continue this exact run later (e.g. after a crash,
# or to train further after bumping total_iterations in config.json), switch this
# cell to `run_training(run_dir)` instead -- re-running with `config` again would
# start a second, separate run.
# model, optimizer, run_dir = run_training(config)
# config = Path('runs/20260718T115422_baseline4layer_0')
model, optimizer, run_dir = run_training(config, save_on_exit=False)

In [ ]:
context_length = config["model_params"]["context_length"]
device = config["model_params"]["device"]

In [ ]:
run_dirs = sorted(rd for rd in Path("runs").iterdir() if rd.is_dir())

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, ax = plt.subplots(figsize=(10, 4))
ax_lr = None

for i, rd in enumerate(run_dirs):
    if not (rd / "train.jsonl").exists():
        continue
    color = colors[i % len(colors)]
    description = json.loads((rd / "config.json").read_text())["description"]
    rows = [json.loads(line) for line in (rd / "train.jsonl").read_text().splitlines()]
    steps = [r["step"] for r in rows]
    losses = [r["loss"] for r in rows]
    val_rows = [r for r in rows if r["val_loss"] is not None]
    val_steps = [r["step"] for r in val_rows]
    val_losses = [r["val_loss"] for r in val_rows]

    ax.plot(steps, losses, color=color, label=f"{description} (train)")
    ax.plot(val_steps, val_losses, color=color, linestyle="--", label=f"{description} (val)")

    if rows[0].get("lr") is not None:
        if ax_lr is None:
            ax_lr = ax.twinx()
            ax_lr.set_ylabel("learning rate")
        lrs = [r["lr"] for r in rows]
        ax_lr.plot(steps, lrs, color=color, linestyle=":", alpha=0.35, label=f"{description} (lr)")

ax.set_xlabel("iteration")
ax.set_ylabel("loss")
ax.set_title("training curves")

handles, labels = ax.get_legend_handles_labels()
if ax_lr is not None:
    lr_handles, lr_labels = ax_lr.get_legend_handles_labels()
    handles += lr_handles
    labels += lr_labels
ax.legend(handles, labels, loc="upper right", fontsize=8)
plt.show()

In [ ]:
import sys


def generate(prompt: str, max_tokens: int = 400, temperature: float = 0.9, width: int = 90) -> str:
    """Stream-decode tokens from `model` and print with word-wrap at `width`.

    Uses the surrounding notebook scope for `model`, `tokenizer`, `context_length`,
    and `device`. Stops early when the <|endoftext|> token is sampled. Returns the
    full generated string (prompt + completion) for any post-hoc use.
    """
    EOT_ID = tokenizer.encode("<|endoftext|>")[0]

    all_ids = tokenizer.encode(prompt)
    model_input = torch.tensor(all_ids, dtype=torch.long).unsqueeze(0).to(device)

    col = 0
    word_buf = ""

    def stream(text: str) -> None:
        nonlocal col, word_buf
        text = text.replace("\n", " ")
        for ch in text:
            if ch == " ":
                if word_buf:
                    space = 1 if col > 0 else 0
                    if col + space + len(word_buf) > width:
                        sys.stdout.write("\n")
                        col = 0
                    elif space:
                        sys.stdout.write(" ")
                        col += 1
                    sys.stdout.write(word_buf)
                    col += len(word_buf)
                    word_buf = ""
            else:
                word_buf += ch
        sys.stdout.flush()

    stream(prompt)
    printed = prompt

    model.eval()
    with torch.no_grad():
        for _ in range(max_tokens):
            logits = model(model_input)[:, -1]
            probs = torch.softmax(logits / temperature, dim=-1)
            pred = torch.multinomial(probs, num_samples=1)
            token_id = pred.item()
            if token_id == EOT_ID:
                break
            all_ids.append(token_id)
            model_input = torch.cat([model_input, pred], dim=1)[:, -context_length:]

            decoded = tokenizer.decode(all_ids)
            new_part = decoded[len(printed) :]
            if new_part:
                stream(new_part)
                printed = decoded

    stream(" ")  # flush trailing partial word
    return tokenizer.decode(all_ids)

In [ ]:
generate("tell me a story about apples", max_tokens=400, temperature=0.9);